# UAV Neo — Sim Core Test

This notebook tests each sensor module and flight command in the UAVNeo Simulator.
Run each cell in order. The simulator must be running and reachable (auto-detects on WSL2, or set `DRONE_SIM_IP`).

Unlike the real-drone notebook, the simulator has no `go_async()` — every test runs inside a single `drone.go()` state machine that cycles through phases on a timer. After running the test cell, click **User Program Mode** in the simulator window.

## 1. Initialize Drone

In [ ]:
import sys, os
_HERE = os.getcwd()
sys.path.insert(0, os.path.join(_HERE, '../../library'))
sys.path.insert(0, os.path.join(_HERE, '../../library/simulation'))

import drone_core
import drone_utils as uav_utils
import numpy as np
import cv2 as cv
from IPython.display import display
import ipywidgets as widgets

drone = drone_core.create_drone(isSimulation=True)

# Force the dot-matrix display module into headless mode so the Rich Live
# panel never starts. This sim notebook does not test the dot matrix, and
# the rendering thread can throttle the update loop in Jupyter.
drone.display._Display__isHeadless = True

print('Drone initialized in simulation mode (display disabled).')

## 2. Helper Functions and Live Widgets

In [ ]:
def to_jpeg_bytes(bgr_image, resize=(320, 240)):
    """Convert a BGR numpy image to JPEG bytes for widget display."""
    if resize:
        bgr_image = cv.resize(bgr_image, resize)
    _, buf = cv.imencode('.jpg', bgr_image, [cv.IMWRITE_JPEG_QUALITY, 70])
    return buf.tobytes()

def colorize_depth(depth_image, max_depth_cm=300):
    """Convert a float32 depth image (cm) to a JET-colored BGR image."""
    valid = depth_image > 0
    normalized = np.clip(depth_image / max_depth_cm, 0.0, 1.0)
    inverted = (255 - (normalized * 255)).astype(np.uint8)
    inverted[~valid] = 0
    colored = cv.applyColorMap(inverted, cv.COLORMAP_JET)
    colored[~valid] = [0, 0, 0]
    return colored

phase_label   = widgets.Label(value='Waiting for User Program Mode...')
status_label  = widgets.Label(value='')
sensor_label  = widgets.Label(value='')
control_label = widgets.Label(value='')
img_widget    = widgets.Image(format='jpeg', width=320, height=240)

display(widgets.VBox([phase_label, status_label, img_widget, sensor_label, control_label]))

## 3. Test Phases

| # | Phase | Duration | Tests |
|--:|-------|---------:|-------|
| 1 | Forward Camera   | 8 s  | `camera.get_color_image_no_copy()` |
| 2 | Downward Camera  | 8 s  | `camera.get_downward_image()` |
| 3 | Depth Camera     | 8 s  | `camera.get_depth_image()` |
| 4 | Physics / IMU    | 8 s  | accel, velocity, gyro, altitude, attitude, GPS |
| 5 | Xbox Controller  | 8 s  | buttons, triggers, joysticks |
| 6 | Telemetry        | 4 s  | `telemetry.declare_variables` + `record` |
| 7 | PCMD / Flight    | 14 s | takeoff → climb → fwd+right → yaw → stop → land |

Total runtime: ~58 s. The notebook auto-exits the simulator when DONE.

## 4. Run All Tests

Blocks until the sequence finishes. The DONE phase sends an exit packet to the simulator so `drone.go()` returns automatically — no need to click Stop.

In [ ]:
PHASES = [
    ('FORWARD_CAM',  8.0),
    ('DOWNWARD_CAM', 8.0),
    ('DEPTH_CAM',    8.0),
    ('PHYSICS',      8.0),
    ('CONTROLLER',   8.0),
    ('TELEMETRY',    4.0),
    ('FLIGHT',      14.0),
    ('DONE',         0.0),
]

state = {'phase_idx': 0, 'phase_t': 0.0, 'frames': 0, 'flight_started': False, 'land_sent': False}

results = {k: False for k in [
    'forward_camera', 'downward_camera', 'depth_camera',
    'physics_imu', 'physics_velocity', 'physics_altitude', 'physics_attitude', 'physics_gps',
    'controller', 'telemetry',
    'flight_takeoff', 'flight_pcmd', 'flight_land', 'delta_time',
]}

class TestComplete(SystemExit):
    """Raised from update() to unwind drone.go() cleanly when all phases finish."""

def start():
    drone.set_update_slow_time(0.5)
    drone.flight.set_max_speed(1.0)
    drone.flight.stop()
    drone.telemetry.declare_variables('altitude_m', 'accel_z')
    state.update(phase_idx=0, phase_t=0.0, frames=0, flight_started=False, land_sent=False)
    phase_label.value = f'Phase 1/{len(PHASES)}: {PHASES[0][0]}'
    status_label.value = 'Starting...'

def _advance_phase():
    state['phase_idx'] += 1
    state.update(phase_t=0.0, frames=0)
    drone.flight.stop()
    phase_label.value = f'Phase {state["phase_idx"]+1}/{len(PHASES)}: {PHASES[state["phase_idx"]][0]}'

def update():
    dt = drone.get_delta_time()
    if dt > 0:
        results['delta_time'] = True
    state['phase_t'] += dt
    state['frames']  += 1
    name, duration = PHASES[state['phase_idx']]
    t = state['phase_t']

    if name == 'FORWARD_CAM':
        img = drone.camera.get_color_image_no_copy()
        if img is not None and img.size > 0:
            results['forward_camera'] = True
            img_widget.value = to_jpeg_bytes(img)
        status_label.value = f'Forward camera | Frame {state["frames"]} | {t:.1f}s / {duration:.0f}s'

    elif name == 'DOWNWARD_CAM':
        img = drone.camera.get_downward_image()
        if img is not None and img.size > 0:
            results['downward_camera'] = True
            img_widget.value = to_jpeg_bytes(img)
        status_label.value = f'Downward camera | Frame {state["frames"]} | {t:.1f}s / {duration:.0f}s'

    elif name == 'DEPTH_CAM':
        depth = drone.camera.get_depth_image()
        if depth is not None and depth.size > 0:
            results['depth_camera'] = True
            img_widget.value = to_jpeg_bytes(colorize_depth(depth))
            cy, cx = depth.shape[0] // 2, depth.shape[1] // 2
            center_dist = float(depth[cy, cx])
            status_label.value = (
                f'Depth | Frame {state["frames"]} | {t:.1f}s / {duration:.0f}s | Center: {center_dist:.0f} cm'
            )

    elif name == 'PHYSICS':
        accel = drone.physics.get_linear_acceleration()
        vel   = drone.physics.get_linear_velocity()
        gyro  = drone.physics.get_angular_velocity()
        alt   = drone.physics.get_altitude()
        att   = drone.physics.get_attitude()
        gps   = drone.physics.get_gps()
        if np.linalg.norm(accel) > 0.5: results['physics_imu'] = True
        results['physics_velocity'] = True
        if alt is not None: results['physics_altitude'] = True
        if np.any(att != 0) or alt == 0: results['physics_attitude'] = True
        if np.any(gps != 0): results['physics_gps'] = True
        sensor_label.value = (
            f'Accel ({accel[0]:+6.2f},{accel[1]:+6.2f},{accel[2]:+6.2f}) m/s^2  '
            f'Vel ({vel[0]:+5.2f},{vel[1]:+5.2f},{vel[2]:+5.2f}) m/s  '
            f'Gyro ({gyro[0]:+5.2f},{gyro[1]:+5.2f},{gyro[2]:+5.2f}) rad/s'
        )
        control_label.value = (
            f'Altitude {alt:.2f} m  Attitude (P:{att[0]:+6.1f}, R:{att[1]:+6.1f}, Y:{att[2]:+6.1f}) deg  '
            f'GPS ({gps[0]:.5f}, {gps[1]:.5f}, {gps[2]:.1f})'
        )
        status_label.value = f'IMU / Physics | {t:.1f}s / {duration:.0f}s'

    elif name == 'CONTROLLER':
        lx, ly = drone.controller.get_joystick(drone.controller.Joystick.LEFT)
        rx, ry = drone.controller.get_joystick(drone.controller.Joystick.RIGHT)
        lt = drone.controller.get_trigger(drone.controller.Trigger.LEFT)
        rt = drone.controller.get_trigger(drone.controller.Trigger.RIGHT)
        pressed = [b.name for b in drone.controller.Button if drone.controller.is_down(b)]
        results['controller'] = True
        sensor_label.value = (
            f'Left joy:  ({lx:+.2f}, {ly:+.2f})   Right joy: ({rx:+.2f}, {ry:+.2f})   '
            f'Triggers:  L={lt:.2f} R={rt:.2f}'
        )
        control_label.value = f'Buttons:   {", ".join(pressed) if pressed else "none"}'
        status_label.value = f'Xbox Controller | {t:.1f}s / {duration:.0f}s'

    elif name == 'TELEMETRY':
        try:
            drone.telemetry.record(
                drone.physics.get_altitude(),
                float(drone.physics.get_linear_acceleration()[2]),
            )
            results['telemetry'] = True
        except Exception as e:
            control_label.value = f'Telemetry error: {e}'
        status_label.value = f'Telemetry | {t:.1f}s / {duration:.0f}s | writing log.csv'

    elif name == 'FLIGHT':
        # Kick off takeoff exactly once. The sim needs continuous PCMD setpoints
        # afterward, so each branch below sends a command every frame.
        if not state['flight_started']:
            drone.flight.takeoff()
            results['flight_takeoff'] = True
            state['flight_started'] = True

        if t < 3.0:
            pitch, roll, yaw, throttle = 0.0, 0.0, 0.0, 1.0
            drone.flight.send_pcmd(pitch, roll, yaw, throttle)
            substep = 'climb'
        elif t < 6.0:
            pitch, roll, yaw, throttle = 0.5, 0.3, 0.0, 0.0
            drone.flight.send_pcmd(pitch, roll, yaw, throttle)
            results['flight_pcmd'] = True
            substep = 'fwd + right'
        elif t < 9.0:
            pitch, roll, yaw, throttle = 0.0, 0.0, 0.5, 0.0
            drone.flight.send_pcmd(pitch, roll, yaw, throttle)
            substep = 'yaw'
        elif t < 11.0:
            pitch, roll, yaw, throttle = 0.0, 0.0, 0.0, 0.0
            drone.flight.stop()
            substep = 'stop'
        else:
            pitch, roll, yaw, throttle = 0.0, 0.0, 0.0, 0.0
            if not state['land_sent']:
                drone.flight.land()
                results['flight_land'] = True
                state['land_sent'] = True
            substep = 'land'

        # Live telemetry + state readout while flying
        accel = drone.physics.get_linear_acceleration()
        vel   = drone.physics.get_linear_velocity()
        alt   = drone.physics.get_altitude()
        att   = drone.physics.get_attitude()
        gps   = drone.physics.get_gps()
        sensor_label.value = (
            f'Vel ({vel[0]:+5.2f},{vel[1]:+5.2f},{vel[2]:+5.2f}) m/s  '
            f'Alt {alt:6.2f} m  '
            f'Att (P:{att[0]:+6.1f}, R:{att[1]:+6.1f}, Y:{att[2]:+6.1f}) deg  '
            f'GPS ({gps[0]:.5f}, {gps[1]:.5f}, {gps[2]:.1f})'
        )
        control_label.value = (
            f'Substep: {substep:12s}  PCMD (p:{pitch:+.2f} r:{roll:+.2f} y:{yaw:+.2f} t:{throttle:+.2f})  '
            f'Accel z {accel[2]:+5.2f} m/s^2'
        )
        status_label.value = f'PCMD / Flight | {t:.1f}s / {duration:.0f}s'

    elif name == 'DONE':
        # End of the test sequence. Send an exit packet to Unity, then raise
        # SystemExit to unwind drone.go() — its inner loop blocks on recvfrom()
        # otherwise, hanging the notebook even after we ask Unity to exit.
        drone.flight.stop()
        try:
            drone._DroneSim__send_header(drone.Header.python_exit, True)
        except Exception:
            pass
        status_label.value = 'All phases complete — sequence finished.'
        raise TestComplete

    if name != 'DONE' and t >= duration:
        _advance_phase()

drone.set_start_update(start, update)
try:
    drone.go()
    print('drone.go() returned (simulator sent exit).')
except TestComplete:
    print('Sequence complete — drone.go() unwound cleanly.')
except SystemExit:
    print('drone.go() exited via SystemExit (simulator likely closed).')

## 5. Summary

In [ ]:
labels = {
    'forward_camera':   'Forward camera',
    'downward_camera':  'Downward (nadir) camera',
    'depth_camera':     'Depth camera',
    'physics_imu':      'IMU acceleration',
    'physics_velocity': 'Linear velocity',
    'physics_altitude': 'Altimeter',
    'physics_attitude': 'Attitude (P/R/Y)',
    'physics_gps':      'GPS',
    'controller':       'Xbox controller',
    'telemetry':        'Telemetry logger',
    'flight_takeoff':   'flight.takeoff()',
    'flight_pcmd':      'flight.send_pcmd()',
    'flight_land':      'flight.land()',
    'delta_time':       'drone.get_delta_time()',
}

print('=' * 60)
print('  UAV Neo Sim Core Test — Results')
print('=' * 60)
all_pass = True
for key, label in labels.items():
    ok = results.get(key, False)
    if not ok:
        all_pass = False
    print(f'  [{"PASS" if ok else "FAIL"}] {label}')
print('=' * 60)
print(f'  Overall: {"ALL PASS" if all_pass else "SOME FAILURES — check above"}')
print('=' * 60)

try:
    drone.telemetry.visualize()
    print('Telemetry plot written to log.png')
except Exception as e:
    print(f'(telemetry.visualize skipped: {e})')